# M0 feasibility: train the stock Go2 velocity task with mjlab on a Kaggle T4

**Status: written without being run.** The author of this notebook had no GPU. Every line
whose behaviour could not be checked against a primary source carries a `# VERIFY:` comment
with the source URL. Treat the first run as the test.

What it does, in order:

1. Prints GPU, driver and CUDA information.
2. Installs a pinned mjlab stack (no viewer, headless).
3. Runs a Warp / MuJoCo Warp smoke test on the GPU. **This is the M0 gate.**
4. Trains the stock flat-ground velocity task for a time-boxed run and records wall-clock time
   and environment steps per second.
5. Exports the policy to ONNX in `faultwalk` contract form (ONNX + metadata JSON), checks
   ONNX/PyTorch parity, and keeps mjlab's own ONNX export too.
6. Evaluates briefly on the GPU (mean reward, velocity-tracking error, falls) and in plain CPU
   MuJoCo with the Menagerie Go2 (sim-to-sim smoke check).
7. Writes everything to `/kaggle/working/m0/` and zips it for download.

Every step records success or the failure reason in `/kaggle/working/m0/report.json`; the last
cell prints a diagnosis instead of a stack trace.

**Kaggle settings:** Accelerator = *GPU T4 x2* (only GPU 0 is used), Internet = *On*,
Persistence off is fine. Either run cells interactively, or use *Save Version -> Save & Run All
(Commit)* so it runs in the background, then download `m0.zip` from the version's Output tab.

In [ ]:
# ---- Parameters -------------------------------------------------------------------------
from pathlib import Path

# "unitree_go2": task Unitree-Go2-Flat from unitreerobotics/unitree_rl_mjlab (pins mjlab 1.2.0).
#                This is the Go2 the project needs; stock mjlab has no Go2 task.
# "mjlab_go1":   task Mjlab-Velocity-Flat-Unitree-Go1 from stock mjlab 1.6.1 (newest release).
#                Go1 only, so it is a throughput/feasibility check; cpu_sim cannot replay it.
STACK = "unitree_go2"

NUM_ENVS = 2048  # parallel environments on the GPU
MAX_ITERATIONS = 400  # PPO iterations (24 env steps x NUM_ENVS samples each)
TIME_BUDGET_MIN = 25  # hard stop for the training subprocess; last saved checkpoint is used
SAVE_INTERVAL = 50  # checkpoint every N iterations
SEED = 42
EVAL_ENVS = 256
EVAL_SECONDS = 20.0
CPU_SIM_SECONDS = 10.0
CPU_SIM_VX = 0.5

# Where the faultwalk package comes from (needed for contract export and the CPU check):
#  - a pip-installable git URL of this repo (push first; a private repo needs a token
#    in the URL), or
#  - a Kaggle Dataset containing the repo's `src/faultwalk/` (found automatically under
#    /kaggle/input).
# VERIFY: repo/branch that holds src/faultwalk once the M0 restructure is pushed.
FAULTWALK_PIP = "git+https://github.com/FrostyZ07/RL-Maze-Solver.git@master"

OUT = Path("/kaggle/working/m0")
OUT.mkdir(parents=True, exist_ok=True)
WORK = Path("/kaggle/working/src_checkout")
WORK.mkdir(parents=True, exist_ok=True)

STACKS = {
    "unitree_go2": dict(
        task="Unitree-Go2-Flat",
        repo="https://github.com/unitreerobotics/unitree_rl_mjlab.git",
        # 2026-04-13, the latest commit on main when this notebook was written
        commit="1425b15f73bd4095f0df53709d7c389c3eb9e790",
        # Pins: unitree setup.py requires mjlab==1.2.0, mujoco-warp==3.5.0
        # https://github.com/unitreerobotics/unitree_rl_mjlab/blob/main/setup.py
        # mujoco / warp-lang pinned to the releases contemporary with mjlab 1.2.0 (2026-03-06).
        pins=[
            "mjlab==1.2.0",
            "mujoco-warp==3.5.0",
            "mujoco==3.5.0",
            "warp-lang==1.12.0",
            "rsl-rl-lib==5.0.1",
        ],  # VERIFY: warp-lang 1.12.0 + mujoco-warp 3.5.0 combination untested
        experiment="go2_velocity",
    ),
    "mjlab_go1": dict(
        task="Mjlab-Velocity-Flat-Unitree-Go1",
        repo=None,
        commit=None,
        # mjlab 1.6.1 (PyPI, 2026-11-15) requires mujoco~=3.11.0, mujoco-warp~=3.11.0,
        # rsl-rl-lib==5.4.2
        # https://pypi.org/project/mjlab/1.6.1/
        pins=[
            "mjlab==1.6.1",
            "mujoco-warp==3.11.0",
            "mujoco==3.11.0",
            "warp-lang==1.16.0",
            "rsl-rl-lib==5.4.2",
        ],  # VERIFY: warp-lang 1.16.0 choice untested
        experiment="go1_velocity",
    ),
}
CFG = STACKS[STACK]
print(STACK, CFG["task"])

In [ ]:
# ---- Report helpers + GPU info ------------------------------------------------------------
import json
import os
import platform
import subprocess
import sys
import time
import traceback

REPORT_PATH = OUT / "report.json"
report = {
    "stack": STACK,
    "task": CFG["task"],
    "params": dict(
        NUM_ENVS=NUM_ENVS,
        MAX_ITERATIONS=MAX_ITERATIONS,
        TIME_BUDGET_MIN=TIME_BUDGET_MIN,
        SAVE_INTERVAL=SAVE_INTERVAL,
        SEED=SEED,
    ),
    "steps": {},
    "failures": [],
}


def save_report():
    REPORT_PATH.write_text(json.dumps(report, indent=2, default=str))


def record(step, ok, **info):
    report["steps"][step] = {"ok": ok, **info}
    if not ok:
        report["failures"].append({"step": step, **info})
    save_report()
    print(("OK   " if ok else "FAIL ") + step, {k: v for k, v in info.items() if k != "traceback"})


def sh(cmd, **kw):
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    return subprocess.run(cmd, shell=isinstance(cmd, str), text=True, capture_output=True, **kw)


gpu = {"python": sys.version, "platform": platform.platform()}
r = sh("nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv,noheader")
gpu["nvidia_smi"] = r.stdout.strip() or r.stderr.strip()
print(sh("nvidia-smi").stdout)
try:
    import torch

    gpu.update(
        torch=torch.__version__,
        torch_cuda=torch.version.cuda,
        cuda_available=torch.cuda.is_available(),
    )
    if torch.cuda.is_available():
        gpu.update(
            device=torch.cuda.get_device_name(0),
            capability=".".join(map(str, torch.cuda.get_device_capability(0))),
        )
except Exception as e:  # torch missing is not fatal here
    gpu["torch_error"] = repr(e)
report["gpu"] = gpu
record("gpu_info", bool(gpu.get("cuda_available")), **gpu)

In [ ]:
# ---- Install the pinned stack -------------------------------------------------------------
t0 = time.time()
try:
    # torch: keep Kaggle's preinstalled CUDA build if it satisfies mjlab (torch>=2.7.0, see
    # https://pypi.org/project/mjlab/1.2.0/ and /1.6.1/). Upgrading torch from plain PyPI could pull
    # a CUDA build newer than the Kaggle driver supports.
    import torch

    major, minor = (int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    if (major, minor) < (2, 7):
        # VERIFY: cu126 wheels need driver >= 525 (CUDA minor-version compatibility);
        # https://download.pytorch.org/whl/cu126
        r = sh(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "torch==2.8.0",
                "--index-url",
                "https://download.pytorch.org/whl/cu126",
            ]
        )
        print(r.stdout[-2000:], r.stderr[-2000:])
    pins = CFG["pins"] + ["onnx", "onnxruntime", "tensorboard"]
    r = sh([sys.executable, "-m", "pip", "install", "-q", *pins])
    print(r.stdout[-3000:], r.stderr[-3000:])
    if r.returncode != 0:
        raise RuntimeError("pip install failed: " + r.stderr[-1500:])
    if CFG["repo"]:
        dst = WORK / "unitree_rl_mjlab"
        if not dst.exists():
            r = sh(["git", "clone", CFG["repo"], str(dst)])
            if r.returncode != 0:
                raise RuntimeError(r.stderr)
        r = sh(["git", "-C", str(dst), "checkout", CFG["commit"]])
        if r.returncode != 0:
            raise RuntimeError(r.stderr)
        # --no-deps: dependencies were pinned above. setup.py installs a top-level package
        # literally named `src` (packages=["src"]).
        r = sh([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(dst)])
        if r.returncode != 0:
            raise RuntimeError(r.stderr)
    # faultwalk: --no-deps so it cannot replace the pinned mujoco with a newer one.
    r = sh([sys.executable, "-m", "pip", "install", "-q", "--no-deps", FAULTWALK_PIP])
    if r.returncode != 0:
        import glob

        hits = glob.glob("/kaggle/input/**/faultwalk/contract.py", recursive=True)
        if not hits:
            print("faultwalk not installable:", r.stderr[-800:])
            report["faultwalk_source"] = None
        else:
            sys.path.insert(0, str(Path(hits[0]).parent.parent))
            report["faultwalk_source"] = hits[0]
    else:
        report["faultwalk_source"] = FAULTWALK_PIP
    versions = sh([sys.executable, "-m", "pip", "freeze"]).stdout
    keep = (
        "mjlab",
        "mujoco",
        "warp",
        "rsl",
        "torch",
        "onnx",
        "tensordict",
        "numpy",
        "faultwalk",
        "unitree",
    )
    report["versions"] = [ln for ln in versions.splitlines() if ln.lower().startswith(keep)]
    record("install", True, seconds=round(time.time() - t0, 1), versions=report["versions"])
except Exception as e:
    record("install", False, error=repr(e), traceback=traceback.format_exc())

In [ ]:
# ---- Gate: does Warp / MuJoCo Warp run on this GPU? ---------------------------------------
# A short GPU step of a small model; failures here mean the stack does not run on this GPU.
gate = {}
try:
    import warp as wp

    wp.init()
    dev = wp.get_cuda_device(0)  # VERIFY: https://nvidia.github.io/warp/ (Device API)
    gate.update(
        warp=wp.config.version,
        device=str(dev.name),
        arch=getattr(dev, "arch", None),
        driver=str(wp.get_cuda_driver_version()),
    )  # mjlab uses this call in sim.py
    import mujoco
    import mujoco_warp as mjw

    xml = (
        '<mujoco><worldbody><body><freejoint/><geom size=".1"/></body>'
        '<geom type="plane" size="1 1 .1"/></worldbody></mujoco>'
    )
    m = mujoco.MjModel.from_xml_string(xml)
    d = mujoco.MjData(m)
    # put_model / put_data(nworld=) / step: read in mujoco_warp v3.5.0 mujoco_warp/_src/io.py and
    # __init__.py (https://github.com/google-deepmind/mujoco_warp/tree/v3.5.0).
    # VERIFY: same API in mujoco-warp 3.11.0 (mjlab_go1 stack); not read.
    mw = mjw.put_model(m)
    dw = mjw.put_data(m, d, nworld=1024)
    t = time.time()
    for _ in range(100):
        mjw.step(mw, dw)
    wp.synchronize()
    gate["step_100x1024_s"] = round(time.time() - t, 3)
    gate["qpos_z_world0"] = float(dw.qpos.numpy()[0, 2])
    record("warp_gate", True, **gate)
except Exception as e:
    record("warp_gate", False, error=repr(e), traceback=traceback.format_exc(), **gate)

In [ ]:
# ---- Train (time-boxed subprocess) --------------------------------------------------------
import re
import signal

LOG_ROOT = (
    (WORK / "unitree_rl_mjlab" if CFG["repo"] else WORK) / "logs" / "rsl_rl" / CFG["experiment"]
)
args = [
    f"--env.scene.num-envs={NUM_ENVS}",
    f"--agent.max-iterations={MAX_ITERATIONS}",
    f"--agent.save-interval={SAVE_INTERVAL}",
    f"--agent.seed={SEED}",
    "--agent.logger=tensorboard",
]  # VERIFY: tyro flag spelling; README uses --env.scene.num-envs
if CFG["repo"]:
    cmd = [sys.executable, "scripts/train.py", CFG["task"], *args]
    cwd = WORK / "unitree_rl_mjlab"
else:
    cmd = ["train", CFG["task"], *args]  # console script from mjlab's pyproject
    cwd = WORK
env = dict(os.environ, WANDB_MODE="disabled", PYTHONUNBUFFERED="1")

train = {"cmd": " ".join(cmd), "iterations": []}
log_file = open(OUT / "train_stdout.log", "w")
t_start = time.time()
killed = False
try:
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    it = sps = None
    # rsl_rl console format, see rsl_rl/utils/logger.py (v5.0.1): "Learning iteration i/N",
    # "Steps per second:", "Total steps:", "Mean reward:", "Iteration time:"
    pat = {
        k: re.compile(p)
        for k, p in dict(
            it=r"Learning iteration (\d+)/(\d+)",
            sps=r"Steps per second:\s+([\d.]+)",
            tot=r"Total steps:\s+(\d+)",
            rew=r"Mean reward:\s+(-?[\d.]+)",
            itt=r"Iteration time:\s+([\d.]+)s",
        ).items()
    }
    cur = {}
    for line in proc.stdout:
        log_file.write(line)
        clean = re.sub(r"\x1b\[[0-9;]*m", "", line)
        for k, p in pat.items():
            mm = p.search(clean)
            if mm:
                cur[k] = mm.group(1)
                if k == "it":
                    if cur.get("sps"):
                        train["iterations"].append(dict(cur, t=round(time.time() - t_start, 1)))
                    cur = {"it": mm.group(1)}
        if "Learning iteration" in clean and int(cur.get("it", 0)) % 25 == 0:
            print(clean.strip(), f"| elapsed {time.time() - t_start:.0f}s", flush=True)
        if time.time() - t_start > TIME_BUDGET_MIN * 60:
            killed = True
            proc.send_signal(signal.SIGINT)
            try:
                proc.wait(timeout=60)
            except subprocess.TimeoutExpired:
                proc.kill()
            break
    if cur.get("sps"):
        train["iterations"].append(dict(cur, t=round(time.time() - t_start, 1)))
    proc.wait()
    train.update(
        returncode=proc.returncode,
        killed_by_time_budget=killed,
        wall_clock_s=round(time.time() - t_start, 1),
    )
    its = train["iterations"]
    if its:
        sps_vals = [float(i["sps"]) for i in its]
        train.update(
            iterations_done=int(its[-1]["it"]) + 1,
            total_env_steps=int(its[-1].get("tot", 0)),
            steps_per_second_mean=round(sum(sps_vals) / len(sps_vals)),
            steps_per_second_median=sorted(sps_vals)[len(sps_vals) // 2],
            first_iteration_wall_s=its[0]["t"],  # includes Warp kernel compilation
            last_mean_reward=its[-1].get("rew"),
        )
    (OUT / "timing.json").write_text(json.dumps({k: v for k, v in train.items()}, indent=2))
    ok = bool(its) and (proc.returncode == 0 or killed)
    record(
        "train",
        ok,
        **{k: v for k, v in train.items() if k != "iterations"},
        **(
            {}
            if ok
            else {
                "error": "no iterations parsed; see train_stdout.log tail",
                "tail": open(OUT / "train_stdout.log").read()[-3000:],
            }
        ),
    )
except Exception as e:
    record("train", False, error=repr(e), traceback=traceback.format_exc())
finally:
    log_file.close()

In [ ]:
# ---- Load the last checkpoint, export ONNX in contract form, evaluate on GPU -------------------
import glob
import shutil
from dataclasses import asdict

exported = {}
try:
    runs = sorted(LOG_ROOT.glob("*/"), key=lambda p: p.stat().st_mtime)
    if not runs:
        raise FileNotFoundError(f"no run directory under {LOG_ROOT}")
    run_dir = runs[-1]
    ckpts = sorted(run_dir.glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
    if not ckpts:
        raise FileNotFoundError(f"no model_*.pt in {run_dir}")
    ckpt = ckpts[-1]
    shutil.copy2(ckpt, OUT / ckpt.name)
    for f in run_dir.glob("*.onnx*"):  # mjlab's own export (metadata in ONNX metadata_props)
        shutil.copy2(f, OUT / f"mjlab_{f.name}")
    for f in run_dir.glob("events.out.tfevents*"):
        shutil.copy2(f, OUT / f.name)
    exported.update(run_dir=str(run_dir), checkpoint=ckpt.name)

    import torch

    if CFG["repo"]:
        os.chdir(WORK / "unitree_rl_mjlab")
        sys.path.insert(0, str(WORK / "unitree_rl_mjlab"))
    import mjlab.tasks  # noqa: F401  (registers tasks)

    if CFG["repo"]:
        import src.tasks  # noqa: F401  (registers Unitree-* tasks)
    from mjlab.envs import ManagerBasedRlEnv
    from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper
    from mjlab.tasks.registry import load_env_cfg, load_rl_cfg, load_runner_cls

    # Same pattern as unitree_rl_mjlab scripts/play.py and mjlab scripts/play.py.
    env_cfg = load_env_cfg(CFG["task"], play=True)
    env_cfg.scene.num_envs = EVAL_ENVS
    agent_cfg = load_rl_cfg(CFG["task"])
    device = "cuda:0"
    genv = RslRlVecEnvWrapper(
        ManagerBasedRlEnv(cfg=env_cfg, device=device), clip_actions=agent_cfg.clip_actions
    )
    runner_cls = load_runner_cls(CFG["task"]) or MjlabOnPolicyRunner
    runner = runner_cls(genv, asdict(agent_cfg), device=device)
    runner.load(
        str(ckpt), load_cfg={"actor": True}, strict=True, map_location=device
    )  # VERIFY: load_cfg kw in mjlab 1.6.1
    # Same call as unitree_rl_mjlab scripts/play.py; the policy takes the wrapper's TensorDict obs.
    # VERIFY: https://github.com/unitreerobotics/unitree_rl_mjlab/blob/main/scripts/play.py
    policy = runner.get_inference_policy(device=device)

    try:
        from faultwalk.export import export_policy

        onnx_path, json_path = export_policy(
            runner,
            genv,
            OUT,
            f"{STACK}_flat",
            task_id=CFG["task"],
            clip_actions=agent_cfg.clip_actions,
            source={
                "stack": STACK,
                "checkpoint": ckpt.name,
                "repo": CFG["repo"],
                "commit": CFG["commit"],
                "versions": report.get("versions"),
            },
        )
        exported.update(onnx=str(onnx_path), meta=str(json_path))
        record("export_contract", True, onnx=onnx_path.name, meta=json_path.name)
    except Exception as e:
        record("export_contract", False, error=repr(e), traceback=traceback.format_exc())

    # GPU evaluation: random commands from the play config, no pushes, no obs noise.
    u = genv.unwrapped
    robot = u.scene["robot"]
    dt = u.step_dt
    n_steps = int(EVAL_SECONDS / dt)
    obs = genv.get_observations()
    rew_sum = lin_err = yaw_err = 0.0
    falls = 0
    with torch.inference_mode():
        for _ in range(n_steps):
            obs, rew, dones, extras = genv.step(policy(obs))
            cmd = u.command_manager.get_command("twist")
            lin_err += (
                (cmd[:, :2] - robot.data.root_link_lin_vel_b[:, :2]).norm(dim=1).mean().item()
            )
            yaw_err += (cmd[:, 2] - robot.data.root_link_ang_vel_b[:, 2]).abs().mean().item()
            rew_sum += rew.mean().item()
            time_outs = extras.get("time_outs", torch.zeros_like(dones)).bool()
            falls += int((dones.bool() & ~time_outs).sum().item())
    ev = dict(
        envs=EVAL_ENVS,
        seconds=EVAL_SECONDS,
        mean_step_reward=rew_sum / n_steps,
        lin_vel_tracking_error=lin_err / n_steps,
        yaw_rate_tracking_error=yaw_err / n_steps,
        terminations=falls,
    )
    (OUT / "eval_gpu.json").write_text(json.dumps(ev, indent=2))
    record("eval_gpu", True, **ev)
except Exception as e:
    record("export_or_eval_gpu", False, error=repr(e), traceback=traceback.format_exc(), **exported)

In [ ]:
# ---- Sim-to-sim smoke check: the exported ONNX in plain CPU MuJoCo (Menagerie Go2) --------------
try:
    if STACK != "unitree_go2":
        raise RuntimeError("cpu_sim only supports the Go2; skipped for " + STACK)
    if "onnx" not in exported:
        raise RuntimeError("no contract export to run")
    from dataclasses import asdict as _asdict

    from faultwalk.cpu_sim import run_policy

    res = {}
    for name, cmd in {
        "stand": (0, 0, 0),
        "forward": (CPU_SIM_VX, 0, 0),
        "turn": (0, 0, 0.5),
    }.items():
        s = run_policy(exported["onnx"], exported["meta"], CPU_SIM_SECONDS, *cmd)
        res[name] = _asdict(s)
        print(name, s)
    (OUT / "eval_cpu_mujoco.json").write_text(json.dumps(res, indent=2))
    record(
        "eval_cpu",
        not any(r["fell"] for r in res.values()),
        **{
            k: dict(fell=v["fell"], lin_err=round(v["lin_vel_tracking_error"], 3))
            for k, v in res.items()
        },
    )
except Exception as e:
    record("eval_cpu", False, error=repr(e), traceback=traceback.format_exc())

In [ ]:
# ---- Package outputs -----------------------------------------------------------------------
save_report()
archive = shutil.make_archive("/kaggle/working/m0", "zip", OUT)
for p in sorted(OUT.iterdir()):
    print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.name}")
print("archive:", archive)

In [ ]:
# ---- Diagnosis (fallback): why did it fail? -----------------------------------------------
HINTS = {
    "gpu_info": "No CUDA GPU visible. In Kaggle: Settings -> Accelerator -> GPU T4 x2.",
    "install": "pip could not install the pinned stack (see error). Check Internet is On and "
    "that the pins still exist on PyPI.",
    "warp_gate": "Warp / MuJoCo Warp did not run on this GPU. This is the M0 gate: if the error "
    "mentions the architecture (sm_75), the driver, or 'CUDA graph', record it and "
    "switch to the fallback stack (MuJoCo Playground, JAX, Go1).",
    "train": "Training failed or produced no iterations. Read the tail of train_stdout.log "
    "(CUDA out of memory -> lower NUM_ENVS; tyro error -> check flag names).",
    "export_or_eval_gpu": "Checkpoint loading or GPU evaluation failed; the raw checkpoint and "
    "mjlab's own ONNX are still in m0.zip.",
    "export_contract": "faultwalk export failed (or faultwalk was not installed). mjlab's own "
    "ONNX (mjlab_*.onnx) is still available.",
    "eval_cpu": "The policy fell in CPU MuJoCo (or could not run). For a short run this can be a "
    "weak policy rather than a contract bug; compare with eval_gpu.json.",
}
if not report["failures"]:
    print("All steps passed.")
for f in report["failures"]:
    print(f"\n[{f['step']}] {f.get('error', '')}")
    print("  ->", HINTS.get(f["step"], ""))
print(
    "\nTiming:",
    json.dumps(
        {
            k: report["steps"].get("train", {}).get(k)
            for k in (
                "wall_clock_s",
                "iterations_done",
                "total_env_steps",
                "steps_per_second_mean",
                "first_iteration_wall_s",
                "killed_by_time_budget",
            )
        },
        indent=1,
    ),
)

## Scaling up

The defaults (2048 envs, up to 400 iterations, 25-minute budget) are sized to finish one Kaggle
session quickly; they will not produce a good walker. The mjlab/unitree configs use
`num_steps_per_env=24` and `max_iterations=10001` with 4096 environments in their README
examples.

- Use the measured `steps_per_second_mean` from `timing.json`: wall time per iteration is about
  `NUM_ENVS * 24 / steps_per_second` seconds plus the PPO update.
- Raise `NUM_ENVS` to 4096 if GPU memory allows (watch `nvidia-smi`); T4 has 15 GB.
- Raise `MAX_ITERATIONS` and `TIME_BUDGET_MIN` together. Kaggle sessions end after 12 hours
  (VERIFY: https://www.kaggle.com/docs/notebooks, session limits), and weekly GPU quota applies.
- For runs longer than a session, resume from the last `model_*.pt` (`--agent.resume`, see
  `RslRlBaseRunnerCfg` in mjlab) and push checkpoints to Hugging Face Hub (planned for M2).